# Research: Why is `proj_gt_k` non-zero?

This notebook investigates the ground-truth projection error seen in the kldmPlus DiffCSP++-style lattice pipeline, e.g.

```text
proj_gt_k=0.081758
```

For a mathematically consistent DiffCSP++ lattice target, `proj_k_to_spacegroup(k_gt, sg)` should be almost identical to `k_gt`. If it is not, one of these is likely happening:

- the `space_group` label does not match the processed MatterGen cell,
- the processed/reduced cell no longer satisfies the raw CSV space-group constraints,
- the k-basis projection convention is mismatched,
- the current full-space VP diffusion is adding/preserving forbidden k-components instead of diffusing on the space-group chart.

The notebook is intentionally diagnostic only. It does not train a model.

## Recommendation Being Tested

If `proj_gt_k` is genuinely non-zero on the data, then using only an auxiliary SG loss is weak and internally conflicted: the MSE target says `predict k_gt`, while the auxiliary says `predict proj(k_gt)`.

The more DiffCSP++-faithful path is:

1. project the clean lattice target `k0` to the requested space-group chart,
2. project lattice noise / noisy state during forward diffusion,
3. project predicted `x0` or `eps`,
4. project the prior and every reverse sampling step,
5. keep the auxiliary loss only as a small diagnostic/stabilizer, not as the main symmetry mechanism.

This notebook measures whether the data supports that change.

In [14]:
from __future__ import annotations

import csv
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from torch.utils.data import Subset

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "kldmPlus").exists():
    raise RuntimeError(f"Could not locate repo root from cwd={Path.cwd()}")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kldmPlus.data import CSPTask, resolve_data_root
from kldmPlus.symmetry.latticeSymmetry import LatticeSymmetry

CONFIG_PATH = ROOT / "configs" / "kldm_plus" / "mp_20" / "mp20_diffcsp_k_x0_soft_lattice_laptop.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text())
cfg

{'experiment_name': 'plus_mp20_diffcsp_k_x0_soft_lattice_laptop',
 'sampler_config': '../../samplers/predictor_corrector.yaml',
 'dataset': {'task': 'csp',
  'name': 'mp20',
  'lattice_representation': 'diffcsp_k',
  'root': None,
  'batch_size': 32,
  'num_workers': 0,
  'pin_memory': False,
  'train_subset_fraction': 0.1,
  'train_subset_seed': 2002,
  'train_subset_strategy': 'balanced_space_group'},
 'time_sampler': {'type': 'uniform'},
 'model': {'eps': 1e-06,
  'lambda_l': 1.0,
  'lattice_parameterization': 'x0',
  'lattice_diffusion_type': 'VP',
  'lattice_representation': 'diffcsp_k',
  'lattice_sg_lambda': 1,
  'lattice_sg_normalize': True,
  'lattice_sg_time_weight': 'alpha_squared',
  'wrapped_normal_K': 3,
  'tdm_n_sigmas': 512,
  'tdm_compute_sigma_norm': True,
  'tdm_velocity_scale': 0.15915494309189535,
  'tdm_sigma_norm_estimator': 'quadrature',
  'tdm_sigma_norm_density_K': 20,
  'tdm_sigma_norm_grid_points': 1025,
  'tdm_sigma_norm_mc_samples': 2000,
  'score_network'

In [15]:
DEVICE = torch.device("cpu")
torch.set_grad_enabled(False)

dataset_cfg = cfg["dataset"]
model_cfg = cfg["model"]

task = CSPTask(
    dataset_name=dataset_cfg["name"],
    lattice_parameterization=model_cfg["lattice_parameterization"],
    lattice_representation=dataset_cfg["lattice_representation"],
)
root = resolve_data_root(dataset_cfg.get("root"))
sym = LatticeSymmetry(eps=float(model_cfg.get("eps", 1e-8))).to(DEVICE)

print("repo", ROOT)
print("data root", root)
print("config", CONFIG_PATH)
print("lambda_l", model_cfg.get("lambda_l"))
print("lattice_sg_lambda", model_cfg.get("lattice_sg_lambda"))

repo /workspace
data root /workspace/data
config /workspace/configs/kldm_plus/mp_20/mp20_diffcsp_k_x0_soft_lattice_laptop.yaml
lambda_l 1.0
lattice_sg_lambda 1


## Helpers

In [16]:
def sg_family(sg: int) -> str:
    sg = int(sg)
    if 195 <= sg <= 230:
        return "cubic"
    if 143 <= sg <= 194:
        return "hex_trig"
    if 75 <= sg <= 142:
        return "tetragonal"
    if 16 <= sg <= 74:
        return "orthorhombic"
    if 3 <= sg <= 15:
        return "monoclinic"
    return "triclinic"


def tensor1(x: torch.Tensor) -> torch.Tensor:
    return torch.as_tensor(x, device=DEVICE).reshape(-1)


def get_structure_id(dataset, i: int) -> str | None:
    try:
        return str(dataset.data.structure_id[i])
    except Exception:
        return None


def describe(values, name: str) -> pd.Series:
    s = pd.Series(np.asarray(values, dtype=float), name=name)
    return s.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])


def load_raw_sg_map(split: str) -> dict[str, int]:
    path = root / "mp_20" / "raw" / f"{split}.csv"
    if not path.exists():
        return {}
    mapping = {}
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            value = row.get("spacegroup.number")
            if value not in {None, ""}:
                mapping[str(row["material_id"])] = int(float(value))
    return mapping

## Collect `proj_gt_k` Diagnostics

This reproduces the metric shape used in training: `mean(abs(k - proj(k, sg)))`, then breaks it down by split, family, and constrained/free dimensions.

In [17]:
MAX_PER_SPLIT = 512
SPLITS = ["train", "val", "test"]

rows: list[dict] = []
samples_by_key: dict[tuple[str, int], object] = {}

for split in SPLITS:
    print(f"loading split={split}")
    dataset = task.fit_dataset(root=root, split=split, download=False)
    n = min(len(dataset), MAX_PER_SPLIT)
    raw_sg_map = load_raw_sg_map(split)
    for i in range(n):
        sample = dataset[i]
        samples_by_key[(split, i)] = sample
        k = torch.as_tensor(sample.l, device=DEVICE).reshape(1, 6).float()
        sg = torch.as_tensor(sample.space_group, device=DEVICE).reshape(1).long()
        mask, bias = sym.mask_bias(sg, k)
        proj = sym.proj_k_to_spacegroup(k, sg)
        diff = k - proj
        constrained = 1.0 - mask
        free = mask
        constrained_count = int(constrained.sum().item())
        free_count = int(free.sum().item())
        sid = get_structure_id(dataset, i)
        rows.append({
            "split": split,
            "idx": i,
            "structure_id": sid,
            "sg": int(sg.item()),
            "raw_csv_sg": raw_sg_map.get(sid),
            "family": sg_family(int(sg.item())),
            "proj_abs_mean_all6": float(diff.abs().mean().item()),
            "proj_mse_all6": float(diff.square().mean().item()),
            "proj_abs_sum": float(diff.abs().sum().item()),
            "constrained_count": constrained_count,
            "free_count": free_count,
            "proj_abs_mean_constrained": float((constrained * diff).abs().sum().item() / max(constrained_count, 1)),
            "proj_abs_mean_free": float((free * diff).abs().sum().item() / max(free_count, 1)),
            **{f"k{j}": float(k[0, j].item()) for j in range(6)},
            **{f"proj_k{j}": float(proj[0, j].item()) for j in range(6)},
            **{f"delta_k{j}": float(diff[0, j].item()) for j in range(6)},
        })

df = pd.DataFrame(rows)
display(df.head())
display(describe(df["proj_abs_mean_all6"], "proj_abs_mean_all6"))

loading split=train
dataset_cache action=load dataset=mp_20 split=train reason=fresh path=/workspace/data/mp_20/processed/train
dataset_cache action=from_cache_path:start dataset=mp_20 split=train
dataset_cache action=from_cache_path:done dataset=mp_20 split=train
dataset_cache action=builder_build:start dataset=mp_20 split=train
dataset_cache action=builder_build:done dataset=mp_20 split=train
loading split=val
dataset_cache action=load dataset=mp_20 split=val reason=fresh path=/workspace/data/mp_20/processed/val
dataset_cache action=from_cache_path:start dataset=mp_20 split=val
dataset_cache action=from_cache_path:done dataset=mp_20 split=val
dataset_cache action=builder_build:start dataset=mp_20 split=val
dataset_cache action=builder_build:done dataset=mp_20 split=val
loading split=test
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from

,split,idx,structure_id,sg,raw_csv_sg,family,proj_abs_mean_all6,proj_mse_all6,proj_abs_sum,constrained_count,...,proj_k2,proj_k3,proj_k4,proj_k5,delta_k0,delta_k1,delta_k2,delta_k3,delta_k4,delta_k5
0,train,0,mp-1221227,8,8,monoclinic,0.040951,0.007979,0.245705,2,...,0.0,-0.429154,-0.537018,2.798224,-0.028813,0.000000,-0.216892,0.000000e+00,0.000000e+00,0.0
1,train,1,mp-974729,139,139,tetragonal,0.139728,0.034203,0.838366,4,...,0.0,0.000000,-0.133216,3.075970,0.271952,0.271952,0.063723,-2.307379e-01,0.000000e+00,0.0
2,train,2,mp-1185360,225,225,cubic,0.163376,0.053384,0.980258,5,...,0.0,0.000000,0.000000,2.270811,0.326753,0.326753,0.326753,7.723102e-09,1.152631e-07,0.0
3,train,3,mp-1188861,62,62,orthorhombic,0.000000,0.000000,0.000000,3,...,0.0,-0.230435,-0.816336,3.198338,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.0
4,train,4,mp-677272,122,122,tetragonal,0.163632,0.053391,0.981790,4,...,0.0,0.000000,0.000878,3.358178,-0.324464,-0.324464,-0.331342,1.520412e-03,0.000000e+00,0.0


count    1536.000000
mean        0.081858
std         0.073693
min         0.000000
50%         0.067593
75%         0.163376
90%         0.163377
95%         0.186222
99%         0.226221
max         0.259987
Name: proj_abs_mean_all6, dtype: float64

In [18]:
summary_by_split = df.groupby("split").agg(
    n=("proj_abs_mean_all6", "size"),
    mean_proj_gt_k=("proj_abs_mean_all6", "mean"),
    median_proj_gt_k=("proj_abs_mean_all6", "median"),
    p95_proj_gt_k=("proj_abs_mean_all6", lambda s: s.quantile(0.95)),
    max_proj_gt_k=("proj_abs_mean_all6", "max"),
)
display(summary_by_split)

summary_by_family = df.groupby(["split", "family"]).agg(
    n=("proj_abs_mean_all6", "size"),
    mean_proj_gt_k=("proj_abs_mean_all6", "mean"),
    median_proj_gt_k=("proj_abs_mean_all6", "median"),
    mean_constrained_abs=("proj_abs_mean_constrained", "mean"),
    constrained_dims=("constrained_count", "mean"),
).sort_values(["split", "mean_proj_gt_k"], ascending=[True, False])
display(summary_by_family)

,n,mean_proj_gt_k,median_proj_gt_k,p95_proj_gt_k,max_proj_gt_k
split,,,,,
test,512,0.080333,0.070223,0.182215,0.259987
train,512,0.086478,0.077708,0.204869,0.248707
val,512,0.078763,0.064019,0.183894,0.235453


n  mean_proj_gt_k  median_proj_gt_k  \
split family                                                
test  cubic         115        0.134963          0.163376   
      hex_trig      109        0.114957          0.145819   
      tetragonal    100        0.059766          0.058208   
      monoclinic     74        0.045379          0.055210   
      orthorhombic   93        0.040267          0.016442   
      triclinic      21        0.000000          0.000000   
train cubic         121        0.136372          0.163376   
      hex_trig      116        0.110992          0.144038   
      tetragonal     78        0.080701          0.080914   
      orthorhombic   93        0.052359          0.038310   
      monoclinic     85        0.043957          0.041486   
      triclinic      19        0.000000          0.000000   
val   cubic         110        0.124760          0.163376   
      hex_trig      105        0.114004          0.151003   
      tetragonal     77        0.072646          0.075125   
      orthorhombic  120        0.046675          0.030448   
      monoclinic     77        0.044651          0.048192   
      triclinic      23        0.000000          0.000000   

                    mean_constrained_abs  constrained_dims  
split family                                                
test  cubic                     0.161956               5.0  
      hex_trig                  0.172435               4.0  
      tetragonal                0.089649               4.0  
      monoclinic                0.136136               2.0  
      orthorhombic              0.080533               3.0  
      triclinic                 0.000000               0.0  
train cubic                     0.163646               5.0  
      hex_trig                  0.166488               4.0  
      tetragonal                0.121051               4.0  
      orthorhombic              0.104719               3.0  
      monoclinic                0.131872               2.0  
      triclinic                 0.000000               0.0  
val   cubic                     0.149712               5.0  
      hex_trig                  0.171007               4.0  
      tetragonal                0.108969               4.0  
      orthorhombic              0.093349               3.0  
      monoclinic                0.133953               2.0  
      triclinic                 0.000000               0.0

## Inspect Worst Cases

If the worst cases are concentrated in high-symmetry families, the data/cell likely violates strict ideal SG constraints after preprocessing/reduction. If they are broad, the issue may be convention or label mismatch.

In [19]:
cols = [
    "split", "idx", "structure_id", "sg", "raw_csv_sg", "family",
    "proj_abs_mean_all6", "proj_abs_mean_constrained",
    "k0", "k1", "k2", "k3", "k4", "k5",
    "proj_k0", "proj_k1", "proj_k2", "proj_k3", "proj_k4", "proj_k5",
    "delta_k0", "delta_k1", "delta_k2", "delta_k3", "delta_k4", "delta_k5",
]
display(df.sort_values("proj_abs_mean_all6", ascending=False)[cols].head(25))

,split,idx,structure_id,sg,raw_csv_sg,family,proj_abs_mean_all6,proj_abs_mean_constrained,k0,k1,...,proj_k2,proj_k3,proj_k4,proj_k5,delta_k0,delta_k1,delta_k2,delta_k3,delta_k4,delta_k5
1117,test,93,mp-1211616,189,189,hex_trig,0.259987,0.389981,0.000000,0.000000,...,0.0,0.0,-4.521165e-01,3.893100,0.388418,0.000000,-0.388418,-7.830883e-01,0.0,0.0
227,train,227,mp-864657,194,194,hex_trig,0.248707,0.373061,0.000000,0.000000,...,0.0,0.0,-4.130404e-01,3.559863,0.388418,0.000000,-0.388418,-7.154067e-01,0.0,0.0
879,val,367,mp-861867,194,194,hex_trig,0.235453,0.353179,0.000000,0.000000,...,0.0,0.0,-3.671260e-01,3.600765,0.388418,0.000000,-0.388418,-6.358809e-01,0.0,0.0
1450,test,426,mp-768310,161,161,hex_trig,0.228114,0.342171,0.326738,0.326738,...,0.0,0.0,1.891581e-05,2.980040,0.715156,0.326738,0.326756,3.350777e-05,0.0,0.0
78,train,78,mp-1221909,166,166,hex_trig,0.228013,0.342019,0.326840,0.326409,...,0.0,0.0,-8.899305e-04,3.218017,0.715258,0.326409,0.326409,-3.337013e-07,0.0,0.0
6,train,6,mp-561310,189,189,hex_trig,0.227715,0.341572,0.000000,0.000000,...,0.0,0.0,-3.403214e-01,3.084130,0.388418,0.000000,-0.388418,-5.894537e-01,0.0,0.0
933,val,421,mp-1225452,166,166,hex_trig,0.226938,0.340407,0.327771,0.322719,...,0.0,0.0,-1.048011e-02,2.813640,0.716189,0.322719,0.322719,2.361604e-07,0.0,0.0
200,train,200,mp-755568,161,161,hex_trig,0.226934,0.340400,0.327775,0.322704,...,0.0,0.0,-1.051996e-02,2.787858,0.716193,0.322704,0.322704,-2.837079e-07,0.0,0.0
470,train,470,mp-1215517,166,166,hex_trig,0.226921,0.340381,0.327786,0.322660,...,0.0,0.0,-1.063401e-02,2.587700,0.716204,0.322660,0.322661,-4.784382e-07,0.0,0.0
817,val,305,mp-867537,166,166,hex_trig,0.226860,0.340290,0.327838,0.322452,...,0.0,0.0,-1.117783e-02,2.826817,0.716256,0.322452,0.322452,-2.880329e-07,0.0,0.0


## Does `sample.l` Equal Recomputed k From `sample.cell`?

This checks whether the lattice transform itself is the source of mismatch. It should be near numerical zero because `DiffCSPKContinuousIntervalLattice` computes `sample.l` from `sample.cell`.

In [20]:
rt_rows = []
for (split, i), sample in samples_by_key.items():
    cell = torch.as_tensor(sample.cell, device=DEVICE).reshape(1, 3, 3).float()
    k_saved = torch.as_tensor(sample.l, device=DEVICE).reshape(1, 6).float()
    k_recomputed = sym.m2v(sym.de_so3(cell))
    rt_rows.append({
        "split": split,
        "idx": i,
        "k_recompute_abs_max": float((k_saved - k_recomputed).abs().max().item()),
        "k_recompute_abs_mean": float((k_saved - k_recomputed).abs().mean().item()),
    })

rt_df = pd.DataFrame(rt_rows)
display(rt_df.groupby("split").agg(
    mean_abs_max=("k_recompute_abs_max", "mean"),
    max_abs_max=("k_recompute_abs_max", "max"),
))

,mean_abs_max,max_abs_max
split,,
test,0.0,0.0
train,0.0,0.0
val,0.0,0.0


## Detect Space Group From Processed Cells

This is the strongest label-mismatch test. We compare the raw CSV/MatterGen-carried `space_group` label against a fresh symmetry detection on the processed cell and fractional coordinates.

If many labels disagree, then `proj_gt_k` is not a diffusion bug. It means the SG label attached to the sample is not the exact SG of the cell being used for training.

In [21]:
try:
    from pymatgen.core import Lattice, Structure
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
    PYMATGEN_OK = True
except Exception as exc:
    PYMATGEN_OK = False
    print("pymatgen unavailable:", type(exc).__name__, exc)


def detect_sample_sg(sample, symprec: float = 0.1, angle_tolerance: float = 5.0) -> int | None:
    if not PYMATGEN_OK:
        return None
    cell = torch.as_tensor(sample.cell).reshape(3, 3).detach().cpu().numpy()
    frac = torch.as_tensor(sample.pos).reshape(-1, 3).detach().cpu().numpy()
    z = torch.as_tensor(sample.atomic_numbers).reshape(-1).detach().cpu().numpy().astype(int).tolist()
    structure = Structure(Lattice(cell), z, frac, coords_are_cartesian=False, to_unit_cell=True)
    try:
        return int(SpacegroupAnalyzer(structure, symprec=symprec, angle_tolerance=angle_tolerance).get_space_group_number())
    except Exception:
        return None


DETECT_MAX_PER_SPLIT = 80
detect_rows = []
for split in SPLITS:
    subset = df[df["split"] == split].sort_values("proj_abs_mean_all6", ascending=False).head(DETECT_MAX_PER_SPLIT)
    for _, row in subset.iterrows():
        sample = samples_by_key[(row["split"], int(row["idx"]))]
        detected = detect_sample_sg(sample)
        detect_rows.append({
            "split": row["split"],
            "idx": int(row["idx"]),
            "structure_id": row["structure_id"],
            "label_sg": int(row["sg"]),
            "detected_sg": detected,
            "label_family": sg_family(int(row["sg"])),
            "detected_family": None if detected is None else sg_family(detected),
            "sg_match": None if detected is None else bool(int(row["sg"]) == int(detected)),
            "family_match": None if detected is None else bool(sg_family(int(row["sg"])) == sg_family(detected)),
            "proj_abs_mean_all6": float(row["proj_abs_mean_all6"]),
        })

detect_df = pd.DataFrame(detect_rows)
display(detect_df.head())
if len(detect_df):
    display(detect_df.groupby("split").agg(
        n=("sg_match", "size"),
        sg_match_rate=("sg_match", "mean"),
        family_match_rate=("family_match", "mean"),
        mean_proj_gt_k=("proj_abs_mean_all6", "mean"),
    ))
    display(detect_df.sort_values("proj_abs_mean_all6", ascending=False).head(30))

,split,idx,structure_id,label_sg,detected_sg,label_family,detected_family,sg_match,family_match,proj_abs_mean_all6
0,train,227,mp-864657,194,194,hex_trig,hex_trig,True,True,0.248707
1,train,78,mp-1221909,166,166,hex_trig,hex_trig,True,True,0.228013
2,train,6,mp-561310,189,189,hex_trig,hex_trig,True,True,0.227715
3,train,200,mp-755568,161,161,hex_trig,hex_trig,True,True,0.226934
4,train,470,mp-1215517,166,166,hex_trig,hex_trig,True,True,0.226921


,n,sg_match_rate,family_match_rate,mean_proj_gt_k
split,,,,
test,80,1.0000,1.0000,0.180436
train,80,1.0000,1.0000,0.186899
val,80,0.9875,0.9875,0.181158


,split,idx,structure_id,label_sg,detected_sg,label_family,detected_family,sg_match,family_match,proj_abs_mean_all6
160,test,93,mp-1211616,189,189,hex_trig,hex_trig,True,True,0.259987
0,train,227,mp-864657,194,194,hex_trig,hex_trig,True,True,0.248707
80,val,367,mp-861867,194,194,hex_trig,hex_trig,True,True,0.235453
161,test,426,mp-768310,161,161,hex_trig,hex_trig,True,True,0.228114
1,train,78,mp-1221909,166,166,hex_trig,hex_trig,True,True,0.228013
2,train,6,mp-561310,189,189,hex_trig,hex_trig,True,True,0.227715
81,val,421,mp-1225452,166,166,hex_trig,hex_trig,True,True,0.226938
3,train,200,mp-755568,161,161,hex_trig,hex_trig,True,True,0.226934
4,train,470,mp-1215517,166,166,hex_trig,hex_trig,True,True,0.226921
82,val,305,mp-867537,166,166,hex_trig,hex_trig,True,True,0.226860


## How Much Would Projecting `k_gt` Change the Lattice?

If projection changes k a lot, training on projected targets is not just cosmetic. It changes the decoded cell. This cell may be more symmetry-compatible, but it may also move away from the reduced processed target.

In [22]:
proj_change_rows = []
for (split, i), sample in samples_by_key.items():
    k = torch.as_tensor(sample.l, device=DEVICE).reshape(1, 6).float()
    sg = torch.as_tensor(sample.space_group, device=DEVICE).reshape(1).long()
    k_proj = sym.proj_k_to_spacegroup(k, sg)
    mat = sym.v2m(k)
    mat_proj = sym.v2m(k_proj)
    vol = torch.det(mat).abs().clamp_min(1e-12)
    vol_proj = torch.det(mat_proj).abs().clamp_min(1e-12)
    proj_change_rows.append({
        "split": split,
        "idx": i,
        "sg": int(sg.item()),
        "family": sg_family(int(sg.item())),
        "k_l2_change": float(torch.linalg.vector_norm(k - k_proj).item()),
        "k_abs_mean_change": float((k - k_proj).abs().mean().item()),
        "log_volume_change": float((vol_proj.log() - vol.log()).item()),
        "fro_cell_change": float(torch.linalg.matrix_norm(mat - mat_proj).item()),
    })

proj_change_df = pd.DataFrame(proj_change_rows)
display(proj_change_df.groupby(["split", "family"]).agg(
    n=("k_l2_change", "size"),
    mean_k_l2_change=("k_l2_change", "mean"),
    mean_abs_k_change=("k_abs_mean_change", "mean"),
    mean_abs_log_volume_change=("log_volume_change", lambda s: s.abs().mean()),
    mean_fro_cell_change=("fro_cell_change", "mean"),
).sort_values(["split", "mean_k_l2_change"], ascending=[True, False]))

n  mean_k_l2_change  mean_abs_k_change  \
split family                                                   
test  cubic         115          0.467526           0.134963   
      hex_trig      109          0.449148           0.114957   
      tetragonal    100          0.249060           0.059766   
      monoclinic     74          0.219944           0.045379   
      orthorhombic   93          0.171051           0.040267   
      triclinic      21          0.000000           0.000000   
train cubic         121          0.472407           0.136372   
      hex_trig      116          0.437716           0.110992   
      tetragonal     78          0.314099           0.080701   
      orthorhombic   93          0.222761           0.052359   
      monoclinic     85          0.213166           0.043957   
      triclinic      19          0.000000           0.000000   
val   hex_trig      105          0.437589           0.114004   
      cubic         110          0.432182           0.124760   
      tetragonal     77          0.292251           0.072646   
      monoclinic     77          0.219071           0.044651   
      orthorhombic  120          0.205116           0.046675   
      triclinic      23          0.000000           0.000000   

                    mean_abs_log_volume_change  mean_fro_cell_change  
split family                                                          
test  cubic                       2.052473e-07              2.712199  
      hex_trig                    1.531128e-07              2.523590  
      tetragonal                  1.597404e-07              1.360756  
      monoclinic                  1.192093e-07              1.313967  
      orthorhombic                5.896373e-08              1.025051  
      triclinic                   0.000000e+00              0.000000  
train cubic                       1.852177e-07              2.589399  
      hex_trig                    1.171540e-07              2.408378  
      tetragonal                  1.772856e-07              1.820462  
      orthorhombic                8.716378e-08              1.326820  
      monoclinic                  1.234167e-07              1.261839  
      triclinic                   0.000000e+00              0.000000  
val   hex_trig                    1.771109e-07              2.510585  
      cubic                       2.210790e-07              2.469391  
      tetragonal                  1.486246e-07              1.607987  
      monoclinic                  9.908305e-08              1.268564  
      orthorhombic                6.357829e-08              1.218928  
      triclinic                   0.000000e+00              0.000000

## Does Current Full-Space VP Forward Diffusion Leave the SG Chart?

This compares the current lattice forward process against a projected forward process. The current process adds unconstrained Gaussian noise in all 6 k dimensions. A DiffCSP++-style process keeps the noisy state on the SG chart by projecting the clean target, noise, and final noisy state.

In [23]:
from kldmPlus.diffusionModels.continuous import ContinuousVPDiffusion

vp = ContinuousVPDiffusion(eps=float(model_cfg.get("eps", 1e-6)), parameterization="x0")

rng = torch.Generator(device="cpu").manual_seed(123)
t_values = torch.tensor([0.001, 0.05, 0.25, 0.5, 0.75, 1.0], dtype=torch.float32)
diff_rows = []

for t_scalar in t_values:
    for (split, i), sample in list(samples_by_key.items())[: min(len(samples_by_key), 512)]:
        k0 = torch.as_tensor(sample.l, device=DEVICE).reshape(1, 6).float()
        sg = torch.as_tensor(sample.space_group, device=DEVICE).reshape(1).long()
        eps = torch.randn(k0.shape, generator=rng, dtype=k0.dtype)
        t = t_scalar.reshape(1)

        x_current, _ = vp.forward_sample(t=t, x0=k0, noise=eps)

        k0_projected = sym.proj_k_to_spacegroup(k0, sg)
        eps_projected = sym.proj_k_to_spacegroup(eps, sg)
        x_projected_raw, _ = vp.forward_sample(t=t, x0=k0_projected, noise=eps_projected)
        x_projected = sym.proj_k_to_spacegroup(x_projected_raw, sg)

        current_sg_error = (x_current - sym.proj_k_to_spacegroup(x_current, sg)).abs().mean()
        projected_sg_error = (x_projected - sym.proj_k_to_spacegroup(x_projected, sg)).abs().mean()

        diff_rows.append({
            "t": float(t_scalar.item()),
            "split": split,
            "family": sg_family(int(sg.item())),
            "current_xt_proj_abs_mean": float(current_sg_error.item()),
            "projected_xt_proj_abs_mean": float(projected_sg_error.item()),
            "xt_l2_difference": float(torch.linalg.vector_norm(x_current - x_projected).item()),
        })

diffusion_df = pd.DataFrame(diff_rows)
display(diffusion_df.groupby(["t", "family"]).agg(
    n=("current_xt_proj_abs_mean", "size"),
    current_xt_proj_abs_mean=("current_xt_proj_abs_mean", "mean"),
    projected_xt_proj_abs_mean=("projected_xt_proj_abs_mean", "mean"),
    xt_l2_difference=("xt_l2_difference", "mean"),
).reset_index())

,t,family,n,current_xt_proj_abs_mean,projected_xt_proj_abs_mean,xt_l2_difference
0,0.001,cubic,121,0.139916,0.0,0.477000
1,0.001,hex_trig,116,0.113747,0.0,0.444273
2,0.001,monoclinic,85,0.045246,0.0,0.217969
3,0.001,orthorhombic,93,0.054706,0.0,0.228697
4,0.001,tetragonal,78,0.083554,0.0,0.319421
5,0.001,triclinic,19,0.000000,0.0,0.000000
6,0.050,cubic,121,0.203485,0.0,0.641592
7,0.050,hex_trig,116,0.168740,0.0,0.616984
8,0.050,monoclinic,85,0.065996,0.0,0.313927
9,0.050,orthorhombic,93,0.089699,0.0,0.355793


## Stage 1: DiffCSP++ Free r-Space Projection Diagnostic

Here we test the hard DiffCSP++ affine chart directly, without changing training.

For each clean lattice vector `k0` and space group `G`, DiffCSP++ defines an affine legal subspace:

```text
r0 = B_G^+ (k0 - c_G)
k_legal = c_G + B_G r0
```

In this implementation, `mask` is the diagonal free-coordinate selector `B_G B_G^+`, and `bias` is `c_G`, so:

```text
r0_full = mask * (k0 - bias)
k_legal = bias + r0_full
```

This equals `proj_k_to_spacegroup(k0, G)`, but phrasing it as `r` makes the training question explicit: can we diffuse only the free coordinates and decode legal `k` without badly changing the structure?


In [24]:
try:
    from pymatgen.analysis.structure_matcher import StructureMatcher
    STRUCTURE_MATCHER_OK = PYMATGEN_OK
except Exception as exc:
    STRUCTURE_MATCHER_OK = False
    print("StructureMatcher unavailable:", type(exc).__name__, exc)


def lengths_angles_volume(cell: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    cell = cell.reshape(3, 3)
    lengths = torch.linalg.norm(cell, dim=1)
    alpha = torch.acos(torch.clamp(torch.dot(cell[1], cell[2]) / (lengths[1] * lengths[2]).clamp_min(1e-12), -1.0, 1.0))
    beta = torch.acos(torch.clamp(torch.dot(cell[0], cell[2]) / (lengths[0] * lengths[2]).clamp_min(1e-12), -1.0, 1.0))
    gamma = torch.acos(torch.clamp(torch.dot(cell[0], cell[1]) / (lengths[0] * lengths[1]).clamp_min(1e-12), -1.0, 1.0))
    angles_deg = torch.rad2deg(torch.stack([alpha, beta, gamma]))
    volume = torch.det(cell).abs().clamp_min(1e-12)
    return lengths, angles_deg, volume


def free_r_projection(k0: torch.Tensor, sg: torch.Tensor) -> dict[str, torch.Tensor]:
    mask, bias = sym.mask_bias(sg.reshape(-1).long(), k0)
    r0_full = mask * (k0 - bias)
    k_legal = bias + r0_full
    k_proj = sym.proj_k_to_spacegroup(k0, sg.reshape(-1).long())
    return {
        "mask": mask,
        "bias": bias,
        "r0_full": r0_full,
        "k_legal": k_legal,
        "k_proj": k_proj,
        "free_dims": mask.sum(dim=1),
        "constrained_dims": (1.0 - mask).sum(dim=1),
        "projection_consistency_abs_max": (k_legal - k_proj).abs().amax(dim=1),
    }


def structure_matcher_metrics(sample, legal_cell: torch.Tensor) -> dict[str, float | bool | None]:
    if not STRUCTURE_MATCHER_OK:
        return {"sm_fit": None, "sm_rms": None, "sm_max_dist": None}
    z = torch.as_tensor(sample.atomic_numbers).reshape(-1).detach().cpu().numpy().astype(int).tolist()
    frac = torch.as_tensor(sample.pos).reshape(-1, 3).detach().cpu().numpy()
    raw_cell = torch.as_tensor(sample.cell).reshape(3, 3).detach().cpu().numpy()
    legal_cell_np = legal_cell.reshape(3, 3).detach().cpu().numpy()
    raw_structure = Structure(Lattice(raw_cell), z, frac, coords_are_cartesian=False, to_unit_cell=True)
    legal_structure = Structure(Lattice(legal_cell_np), z, frac, coords_are_cartesian=False, to_unit_cell=True)
    matcher = StructureMatcher(primitive_cell=False, scale=False, attempt_supercell=False)
    rms = matcher.get_rms_dist(raw_structure, legal_structure)
    return {
        "sm_fit": bool(matcher.fit(raw_structure, legal_structure)),
        "sm_rms": None if rms is None else float(rms[0]),
        "sm_max_dist": None if rms is None else float(rms[1]),
    }


In [25]:
R_SPACE_MAX_PER_SPLIT = None  # Set to e.g. 128 for a faster smoke test.
r_space_rows = []

for split in SPLITS:
    split_df = df[df["split"] == split].sort_values("proj_abs_mean_all6", ascending=False)
    if R_SPACE_MAX_PER_SPLIT is not None:
        split_df = split_df.head(int(R_SPACE_MAX_PER_SPLIT))
    print(f"r_space_projection split={split} n={len(split_df)}")
    for _, row in split_df.iterrows():
        key = (str(row["split"]), int(row["idx"]))
        sample = samples_by_key[key]
        k0 = torch.as_tensor(sample.l, device=DEVICE).reshape(1, 6).float()
        sg = torch.as_tensor(sample.space_group, device=DEVICE).reshape(1).long()
        proj = free_r_projection(k0, sg)
        k_legal = proj["k_legal"]

        cell0 = sym.v2m(k0).reshape(3, 3)
        cell_legal = sym.v2m(k_legal).reshape(3, 3)
        lengths0, angles0, volume0 = lengths_angles_volume(cell0)
        lengths_legal, angles_legal, volume_legal = lengths_angles_volume(cell_legal)
        matcher_stats = structure_matcher_metrics(sample, cell_legal)

        r_space_rows.append({
            "split": str(row["split"]),
            "idx": int(row["idx"]),
            "structure_id": row["structure_id"],
            "sg": int(sg.item()),
            "family": sg_family(int(sg.item())),
            "free_dims": int(proj["free_dims"].item()),
            "constrained_dims": int(proj["constrained_dims"].item()),
            "proj_abs_mean_all6": float((k0 - k_legal).abs().mean().item()),
            "proj_l2": float(torch.linalg.vector_norm(k0 - k_legal).item()),
            "r_norm": float(torch.linalg.vector_norm(proj["r0_full"]).item()),
            "projection_consistency_abs_max": float(proj["projection_consistency_abs_max"].item()),
            "length_mae": float((lengths0 - lengths_legal).abs().mean().item()),
            "length_max_abs": float((lengths0 - lengths_legal).abs().max().item()),
            "angle_mae_deg": float((angles0 - angles_legal).abs().mean().item()),
            "angle_max_abs_deg": float((angles0 - angles_legal).abs().max().item()),
            "volume_rel_error": float(((volume_legal - volume0) / volume0.clamp_min(1e-12)).item()),
            "fro_cell_change": float(torch.linalg.matrix_norm(cell0 - cell_legal).item()),
            **matcher_stats,
        })

r_space_df = pd.DataFrame(r_space_rows)
display(r_space_df.head())
display(r_space_df.groupby(["split", "family"]).agg(
    n=("proj_abs_mean_all6", "size"),
    mean_proj_abs=("proj_abs_mean_all6", "mean"),
    p95_proj_abs=("proj_abs_mean_all6", lambda s: s.quantile(0.95)),
    mean_length_mae=("length_mae", "mean"),
    p95_length_max_abs=("length_max_abs", lambda s: s.quantile(0.95)),
    mean_angle_mae_deg=("angle_mae_deg", "mean"),
    p95_angle_max_abs_deg=("angle_max_abs_deg", lambda s: s.quantile(0.95)),
    mean_abs_volume_rel_error=("volume_rel_error", lambda s: s.abs().mean()),
    p95_abs_volume_rel_error=("volume_rel_error", lambda s: s.abs().quantile(0.95)),
    mean_fro_cell_change=("fro_cell_change", "mean"),
    sm_fit_rate=("sm_fit", lambda s: pd.Series(s).dropna().mean() if pd.Series(s).dropna().shape[0] else np.nan),
    mean_sm_rms=("sm_rms", "mean"),
).sort_values(["split", "mean_proj_abs"], ascending=[True, False]))


r_space_projection split=train n=512
r_space_projection split=val n=512
r_space_projection split=test n=512


,split,idx,structure_id,sg,family,free_dims,constrained_dims,proj_abs_mean_all6,proj_l2,r_norm,projection_consistency_abs_max,length_mae,length_max_abs,angle_mae_deg,angle_max_abs_deg,volume_rel_error,fro_cell_change,sm_fit,sm_rms,sm_max_dist
0,train,227,mp-864657,194,hex_trig,2,4,0.248707,0.901967,3.583745,0.0,2.864960,4.667773,19.999994,29.999992,-6.408398e-08,7.347680,False,NaN,NaN
1,train,78,mp-1221909,166,hex_trig,2,4,0.228013,0.851281,3.218017,0.0,0.465448,0.783433,39.981289,60.000019,-1.158493e-07,5.987104,False,NaN,NaN
2,train,6,mp-561310,189,hex_trig,2,4,0.227715,0.805725,3.102849,0.0,1.866263,2.869411,19.999994,30.000008,-1.460850e-07,5.021608,False,NaN,NaN
3,train,200,mp-755568,161,hex_trig,2,4,0.226934,0.849240,2.787878,0.0,0.355542,0.597771,39.779068,60.000004,6.101084e-08,4.643720,False,NaN,NaN
4,train,470,mp-1215517,166,hex_trig,2,4,0.226921,0.849216,2.587722,0.0,0.316662,0.532395,39.776699,60.000004,-1.725839e-07,4.136652,False,NaN,NaN


n  mean_proj_abs  p95_proj_abs  mean_length_mae  \
split family                                                            
test  cubic         115       0.134963      0.163377         0.505671   
      hex_trig      109       0.114957      0.223981         0.431822   
      tetragonal    100       0.059766      0.147623         0.343759   
      monoclinic     74       0.045379      0.091789         0.136880   
      orthorhombic   93       0.040267      0.159163         0.129899   
      triclinic      21       0.000000      0.000000         0.000000   
train cubic         121       0.136372      0.163377         0.481420   
      hex_trig      116       0.110992      0.226678         0.301632   
      tetragonal     78       0.080701      0.163971         0.361492   
      orthorhombic   93       0.052359      0.162230         0.171751   
      monoclinic     85       0.043957      0.097963         0.124904   
      triclinic      19       0.000000      0.000000         0.000000   
val   cubic         110       0.124760      0.163377         0.459333   
      hex_trig      105       0.114004      0.223811         0.405638   
      tetragonal     77       0.072646      0.164287         0.347134   
      orthorhombic  120       0.046675      0.160505         0.154715   
      monoclinic     77       0.044651      0.096677         0.129073   
      triclinic      23       0.000000      0.000000         0.000000   

                    p95_length_max_abs  mean_angle_mae_deg  \
split family                                                 
test  cubic                   0.895868           24.599501   
      hex_trig                1.986687           16.181254   
      tetragonal              2.352615            6.283204   
      monoclinic              0.532583            7.392564   
      orthorhombic            0.729006            6.305137   
      triclinic               0.000000            0.000000   
train cubic                   0.808555           24.432219   
      hex_trig                1.496771           17.037036   
      tetragonal              2.055566            9.607969   
      orthorhombic            0.821770            8.239262   
      monoclinic              0.509739            7.015563   
      triclinic               0.000000            0.000000   
val   cubic                   0.827112           22.430512   
      hex_trig                1.840678           16.420426   
      tetragonal              1.950364            8.432243   
      orthorhombic            0.690626            7.592262   
      monoclinic              0.526739            7.284726   
      triclinic               0.000000            0.000000   

                    p95_angle_max_abs_deg  mean_abs_volume_rel_error  \
split family                                                           
test  cubic                     30.000046               2.010352e-07   
      hex_trig                  60.000019               1.617048e-07   
      tetragonal                25.504061               1.322906e-07   
      monoclinic                29.782364               9.793815e-08   
      orthorhombic              29.310747               6.691842e-08   
      triclinic                  0.000000               0.000000e+00   
train cubic                     30.000038               1.974878e-07   
      hex_trig                  60.000021               1.358221e-07   
      tetragonal                29.008267               1.656893e-07   
      orthorhombic              29.733916               8.254144e-08   
      monoclinic                29.752206               1.159101e-07   
      triclinic                  0.000000               0.000000e+00   
val   cubic                     30.000035               2.215550e-07   
      hex_trig                  60.000025               1.642474e-07   
      tetragonal                27.903433               1.562827e-07   
      orthorhombic              29.768819               7.497293e-08   
      monoclinic        

In [26]:
display(r_space_df.sort_values("proj_abs_mean_all6", ascending=False)[[
    "split", "idx", "structure_id", "sg", "family", "free_dims", "constrained_dims",
    "proj_abs_mean_all6", "proj_l2", "length_mae", "length_max_abs",
    "angle_mae_deg", "angle_max_abs_deg", "volume_rel_error",
    "fro_cell_change", "sm_fit", "sm_rms", "sm_max_dist",
]].head(40))

if "sm_rms" in r_space_df:
    display(r_space_df.sort_values("sm_rms", ascending=False, na_position="last")[[
        "split", "idx", "structure_id", "sg", "family",
        "proj_abs_mean_all6", "length_mae", "angle_mae_deg",
        "volume_rel_error", "sm_fit", "sm_rms", "sm_max_dist",
    ]].head(40))


,split,idx,structure_id,sg,family,free_dims,constrained_dims,proj_abs_mean_all6,proj_l2,length_mae,length_max_abs,angle_mae_deg,angle_max_abs_deg,volume_rel_error,fro_cell_change,sm_fit,sm_rms,sm_max_dist
1024,test,93,mp-1211616,189,hex_trig,2,4,0.259987,0.956538,3.736814,6.256058,20.000000,30.000008,-1.439270e-07,9.415921,False,NaN,NaN
0,train,227,mp-864657,194,hex_trig,2,4,0.248707,0.901967,2.864960,4.667773,19.999994,29.999992,-6.408398e-08,7.347680,False,NaN,NaN
512,val,367,mp-861867,194,hex_trig,2,4,0.235453,0.840287,2.669817,4.199032,20.000008,30.000015,1.791035e-07,7.038651,False,NaN,NaN
1025,test,426,mp-768310,161,hex_trig,2,4,0.228114,0.851455,0.406441,0.684226,39.999191,59.998802,-8.747302e-08,5.221181,False,NaN,NaN
1,train,78,mp-1221909,166,hex_trig,2,4,0.228013,0.851281,0.465448,0.783433,39.981289,60.000019,-1.158493e-07,5.987104,False,NaN,NaN
2,train,6,mp-561310,189,hex_trig,2,4,0.227715,0.805725,1.866263,2.869411,19.999994,30.000008,-1.460850e-07,5.021608,False,NaN,NaN
513,val,421,mp-1225452,166,hex_trig,2,4,0.226938,0.849248,0.360905,0.606791,39.779907,60.000004,1.166927e-07,4.713471,False,NaN,NaN
3,train,200,mp-755568,161,hex_trig,2,4,0.226934,0.849240,0.355542,0.597771,39.779068,60.000004,6.101084e-08,4.643720,False,NaN,NaN
4,train,470,mp-1215517,166,hex_trig,2,4,0.226921,0.849216,0.316662,0.532395,39.776699,60.000004,-1.725839e-07,4.136652,False,NaN,NaN
514,val,305,mp-867537,166,hex_trig,2,4,0.226860,0.849102,0.363108,0.610445,39.765301,59.999985,0.000000e+00,4.747495,False,NaN,NaN


,split,idx,structure_id,sg,family,proj_abs_mean_all6,length_mae,angle_mae_deg,volume_rel_error,sm_fit,sm_rms,sm_max_dist
193,train,124,mp-1219977,187,hex_trig,1.356502e-01,2.003905e-01,19.999990,0.000000e+00,False,2.886125e-01,3.534767e-01
687,val,50,mp-1219722,187,hex_trig,1.324147e-01,2.056782e-01,20.000006,0.000000e+00,False,2.883361e-01,3.531382e-01
1206,test,392,mp-1206749,187,hex_trig,1.315580e-01,1.887569e-01,20.000006,1.403975e-07,False,2.882651e-01,3.530512e-01
1208,test,452,mp-11833,187,hex_trig,1.311527e-01,1.928335e-01,19.999985,1.315266e-07,False,2.882318e-01,3.530104e-01
1199,test,114,mp-1223742,187,hex_trig,1.372903e-01,2.787115e-01,20.000000,2.030806e-07,False,2.875059e-01,3.521214e-01
1202,test,112,mp-1220596,160,hex_trig,1.361962e-01,4.282156e-03,22.370893,-1.132488e-07,False,2.634289e-01,3.820134e-01
1201,test,245,mp-1184509,187,hex_trig,1.363842e-01,1.973768e-01,20.000000,0.000000e+00,True,1.544297e-01,1.544297e-01
373,train,74,mp-21437,136,tetragonal,1.233479e-07,9.536743e-07,0.000008,1.517341e-07,True,8.982009e-06,2.459395e-05
1360,test,173,mp-1271286,127,tetragonal,1.184914e-03,1.658758e-03,0.167089,1.338024e-07,True,8.735256e-06,1.187005e-05
391,train,330,mp-18966,194,hex_trig,5.181282e-08,7.947286e-08,0.000000,1.472632e-07,True,2.352455e-06,4.074442e-06


## Interpret the Result

Use this checklist after running the notebook:

- If `k_recompute_abs_max` is tiny, the transform is not the bug.
- If `sg_match_rate` or `family_match_rate` is low, the attached SG labels do not match the processed cells. Fix labels or detect SG after preprocessing before changing the diffusion math.
- If `projection_consistency_abs_max` is tiny in the r-space section, the `r -> k_legal` construction is equivalent to the current DiffCSP++ projector.
- If r-space projection gives low `length_mae`, low `angle_mae_deg`, low absolute `volume_rel_error`, and acceptable `sm_rms`, then hard free-r diffusion is feasible: train/diffuse only the legal SG coordinates and decode `k = c_G + B_G r`.
- If r-space projection gives large geometry errors or poor StructureMatcher results, then hard r-space changes the physical structures too much. In that case use it only as an explicit approximate hybrid or add a richer preprocessing chart such as pyxtal/conventional-cell handling.
- If current `x_t` leaves the SG chart but projected `x_t` does not, then full-space VP diffusion is mathematically different from DiffCSP++ constrained diffusion.

My default recommendation: use this notebook as the gate. If legal r-space projection barely changes decoded geometry and matcher RMSE, implement free-r DiffCSP++ diffusion. If it changes structures materially, keep the projected hybrid as an ablation rather than pretending it is physically exact.
